<a href="https://colab.research.google.com/github/emad123cde/Hydrogen-price-project/blob/main/LSTM_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------------------------------------------------------
# 1. PyTorch Dataset and DataLoader Setup
# ---------------------------------------------------------
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Tuning parameter: batch_size
# Lower batch_size (32) can help generalization; higher (128) is faster.
train_loader = DataLoader(StockDataset(x_train, y_train), batch_size=64, shuffle=True)
valid_loader = DataLoader(StockDataset(x_valid, y_valid), batch_size=64, shuffle=False)
test_loader = DataLoader(StockDataset(x_test, y_test), batch_size=64, shuffle=False)

# ---------------------------------------------------------
# 2. LSTM Model Definition (Architecture)
# ---------------------------------------------------------
class LSTMModel(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=128, num_layers=2, output_dim=5, dropout=0.3):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Layer Normalization stabilizes training
        self.layer_norm = nn.LayerNorm(input_dim)

        # Stacked LSTM
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.layer_norm(x)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(device)

        out, _ = self.lstm(x, (h0, c0))
        # Use the output of the last time step
        out = self.fc(self.dropout(out[:, -1, :]))
        return out

# ---------------------------------------------------------
# 3. Model Training (Logic & Loop)
# ---------------------------------------------------------
model = LSTMModel(input_dim=5, hidden_dim=128, num_layers=2, output_dim=5).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

# Tuning parameters for the loop
epochs = 100
patience = 15
best_val_loss = float('inf')
counter = 0

train_losses, val_losses = [], []

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    # Validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in valid_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            total_val_loss += criterion(outputs, batch_y).item()

    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(valid_loader)
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

    # Early Stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        torch.save(model.state_dict(), 'best_lstm_model.pt')
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered.")
            break

# ---------------------------------------------------------
# 4. Evaluation and Inverse Scaling
# ---------------------------------------------------------
model.load_state_dict(torch.load('best_lstm_model.pt'))
model.eval()

preds, actuals = [], []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        preds.append(outputs.cpu().numpy())
        actuals.append(batch_y.numpy())

preds = np.concatenate(preds)
actuals = np.concatenate(actuals)

# Transform back to original price scale
# Assuming 'scaler' is the object used by your other team
inv_preds = scaler.inverse_transform(preds)
inv_actuals = scaler.inverse_transform(actuals)

# Visualization of results
plt.figure(figsize=(12,6))
plt.plot(inv_actuals[:, 1], label='Actual Close Price', color='blue') # 1 is 'close' index
plt.plot(inv_preds[:, 1], label='Predicted Close Price', color='orange')
plt.title('EQIX Price Prediction - Test Set')
plt.legend()
plt.show()